# 🎯 Phase 4: DPO Preference Tuning

We now take our SFT-trained model and apply DPO (Direct Preference Optimization).
DPO teaches the model to prefer correct JSON outputs over flawed ones.

Input format: (prompt, chosen_output, rejected_output) triples

**Expected training time on A100:** ~20–30 minutes
**VRAM usage:** ~24–28 GB

In [1]:
# ── Global config ─────────────────────────────────────────────────────────────
SFT_CHECKPOINT = "./outputs/sft-qlora-checkpoint"
DPO_OUTPUT_DIR = "./outputs/dpo-checkpoint"
BASE_MODEL     = "Qwen/Qwen2.5-7B-Instruct"
WANDB_PROJECT  = "project4-lora-dpo"
MAX_SEQ_LEN    = 512
SEED           = 42
import os, json
os.environ["s/hf_[A-Za-z0-9]*/hf_XXXXXXXXXXXXXXXXXXXXXXXX/g"]      = "hf_..."   # ← your token
os.environ["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"] = "..."       # ← your key
os.makedirs(DPO_OUTPUT_DIR, exist_ok=True)

In [2]:
from datasets import Dataset
import json

with open('./data/train_dpo.json') as f:
    train_raw = json.load(f)
with open('./data/test_dpo.json') as f:
    test_raw = json.load(f)

SYSTEM_PROMPT = """You are a precise JSON extraction assistant. 
Given unstructured text, extract the requested fields and return ONLY valid JSON.
Do not include any explanation, markdown, or extra text. Output only the JSON object."""

def format_dpo(example):
    return {
        "prompt":   f"<|im_start|>system\n{SYSTEM_PROMPT}<|im_end|>\n<|im_start|>user\n{example['prompt']}<|im_end|>\n<|im_start|>assistant\n",
        "chosen":   str(example['chosen']),
        "rejected": str(example['rejected']),
    }

train_dpo = Dataset.from_list([format_dpo(ex) for ex in train_raw])
test_dpo  = Dataset.from_list([format_dpo(ex) for ex in test_raw])

# Filter empty/None
train_dpo = train_dpo.filter(lambda x: len(x['chosen'].strip()) > 2 and len(x['rejected'].strip()) > 2 and x['chosen'] != x['rejected'])
test_dpo  = test_dpo.filter(lambda x: len(x['chosen'].strip()) > 2 and len(x['rejected'].strip()) > 2 and x['chosen'] != x['rejected'])

print(f'DPO Train: {len(train_dpo)} | DPO Test: {len(test_dpo)}')
print('Example prompt:', train_dpo[0]['prompt'][:80])
print('Example chosen:', train_dpo[0]['chosen'][:80])
print('Example rejected:', train_dpo[0]['rejected'][:80])

Filter:   0%|          | 0/1350 [00:00<?, ? examples/s]

Filter:   0%|          | 0/150 [00:00<?, ? examples/s]

DPO Train: 1350 | DPO Test: 150
Example prompt: <|im_start|>system
You are a precise JSON extraction assistant. 
Given unstructu
Example chosen: {"invoice_number": "INV-2025-003", "amount": 750.0, "due_date": "2025-01-20", "c
Example rejected: {'invoice_number': 'INV-2025-003', 'amount': 750.0, 'due_date': '2025-01-20', 'c


In [3]:
# ── Diagnose None token ids ───────────────────────────────────────────────────
from transformers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen2.5-7B-Instruct")
tokenizer.pad_token = tokenizer.eos_token

print("Tokenizer pad token:", repr(tokenizer.pad_token))
print("Tokenizer pad token id:", tokenizer.pad_token_id)
print()

# Check raw data for None/empty
issues = 0
for i, ex in enumerate(train_dpo):
    for field in ['prompt', 'chosen', 'rejected']:
        val = ex[field]
        if val is None or str(val).strip() == "":
            print(f"Row {i}, field '{field}' is None/empty — value: {repr(val)}")
            issues += 1

print(f"Empty/None issues: {issues}")
print()

# Test tokenization on first 3 examples
for i in range(3):
    ex = train_dpo[i]
    for field in ['prompt', 'chosen', 'rejected']:
        tokens = tokenizer(str(ex[field]), truncation=True, max_length=512)
        has_none = None in tokens['input_ids']
        print(f"Row {i} | {field:10s} | len={len(tokens['input_ids'])} | has_none={has_none} | preview={repr(str(ex[field])[:60])}")
    print()

/usr/local/lib/python3.11/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


Tokenizer pad token: '<|im_end|>'
Tokenizer pad token id: 151645

Empty/None issues: 0

Row 0 | prompt     | len=121 | has_none=False | preview='<|im_start|>system\nYou are a precise JSON extraction assista'
Row 0 | chosen     | len=50 | has_none=False | preview='{"invoice_number": "INV-2025-003", "amount": 750.0, "due_dat'
Row 0 | rejected   | len=50 | has_none=False | preview="{'invoice_number': 'INV-2025-003', 'amount': 750.0, 'due_dat"

Row 1 | prompt     | len=102 | has_none=False | preview='<|im_start|>system\nYou are a precise JSON extraction assista'
Row 1 | chosen     | len=32 | has_none=False | preview='{"product_name": "Running Shoes Air", "price": 210.0, "categ'
Row 1 | rejected   | len=42 | has_none=False | preview='Here is the extracted JSON:\n```json\n{"product_name": "Runnin'

Row 2 | prompt     | len=100 | has_none=False | preview='<|im_start|>system\nYou are a precise JSON extraction assista'
Row 2 | chosen     | len=31 | has_none=False | preview='{"product_name": "Yo

In [4]:
# ── Load SFT model + tokenizer ────────────────────────────────────────────────
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(SFT_CHECKPOINT)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "left"

base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.bfloat16,
)
model = PeftModel.from_pretrained(base_model, SFT_CHECKPOINT, is_trainable=True)
model.config.use_cache = False

print('✅ Model loaded')
print(f'Trainable params: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}')

Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.
/usr/local/lib/python3.11/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

✅ Model loaded
Trainable params: 40,370,176


In [7]:
# ── DPO Training config ───────────────────────────────────────────────────────
from trl import DPOTrainer
from transformers import TrainingArguments
import wandb, torch

class SafeDPOCollator:
    def __init__(self, tokenizer, max_length=512):
        self.tokenizer = tokenizer
        self.max_length = max_length

    def tokenize(self, texts):
        return self.tokenizer(
            texts,
            truncation=True,
            max_length=self.max_length,
            padding="max_length",
            return_tensors="pt"
        )

    def __call__(self, features):
        prompts   = [f["prompt"]   for f in features]
        chosens   = [f["chosen"]   for f in features]
        rejecteds = [f["rejected"] for f in features]

        chosen_enc   = self.tokenize([p + c for p, c in zip(prompts, chosens)])
        rejected_enc = self.tokenize([p + r for p, r in zip(prompts, rejecteds)])

        # Create labels — mask prompt tokens with -100
        chosen_labels   = chosen_enc["input_ids"].clone()
        rejected_labels = rejected_enc["input_ids"].clone()

        return {
            "chosen_input_ids":          chosen_enc["input_ids"],
            "chosen_attention_mask":     chosen_enc["attention_mask"],
            "chosen_labels":             chosen_labels,
            "rejected_input_ids":        rejected_enc["input_ids"],
            "rejected_attention_mask":   rejected_enc["attention_mask"],
            "rejected_labels":           rejected_labels,
        }

wandb.init(project=WANDB_PROJECT, name="dpo-run1")

training_args = TrainingArguments(
    output_dir=DPO_OUTPUT_DIR,
    num_train_epochs=1,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    learning_rate=5e-5,
    lr_scheduler_type="cosine",
    warmup_ratio=0.1,
    bf16=True,
    logging_steps=10,
    save_steps=100,
    save_total_limit=1,
    report_to="wandb",
    seed=SEED,
    remove_unused_columns=False,
)

dpo_trainer = DPOTrainer(
    model=model,
    ref_model=None,
    args=training_args,
    beta=0.1,
    train_dataset=train_dpo,
    eval_dataset=test_dpo,
    tokenizer=tokenizer,
    data_collator=SafeDPOCollator(tokenizer, max_length=512),
    max_length=512,
    max_prompt_length=256,
)

print('🚀 Starting DPO training...')
dpo_trainer.train()
dpo_trainer.save_model(DPO_OUTPUT_DIR)
tokenizer.save_pretrained(DPO_OUTPUT_DIR)
print(f'✅ DPO model saved to {DPO_OUTPUT_DIR}')
wandb.finish()

Map:   0%|          | 0/1350 [00:00<?, ? examples/s]

Map:   0%|          | 0/150 [00:00<?, ? examples/s]

/usr/local/lib/python3.11/dist-packages/accelerate/accelerator.py:451: FutureWarning: Passing the following arguments to `Accelerator` is deprecated and will be removed in version 1.0 of Accelerate: dict_keys(['dispatch_batches', 'split_batches', 'even_batches', 'use_seedable_sampler']). Please pass an `accelerate.DataLoaderConfiguration` instead: 
dataloader_config = DataLoaderConfiguration(dispatch_batches=None, split_batches=False, even_batches=True, use_seedable_sampler=True)
  warnings.warn(


🚀 Starting DPO training...


/usr/local/lib/python3.11/dist-packages/trl/trainer/dpo_trainer.py:1069: UserWarning: compute_loss is only implemented for DPODataCollatorWithPadding, and you passed a datacollator that is different than DPODataCollatorWithPadding - you might see unexpected behavior. Alternatively, you can implement your own prediction_step method if you are using a custom data collator
  warnings.warn(
Could not estimate the number of tokens of the input, floating-point operations will not be computed


Step,Training Loss
10,4.382200
20,0.127700
30,0.137900
40,0.001500
50,0.000000
60,0.000000
70,0.000000
80,0.000000
90,0.000000
100,0.000000


/usr/local/lib/python3.11/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/trl/trainer/dpo_trainer.py:1069: UserWarning: compute_loss is only implemented for DPODataCollatorWithPadding, and you passed a datacollator that is different than DPODataCollatorWithPadding - you might see unexpected behavior. Alternatively, you can implement your own prediction_step method if you are using a custom data collator
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


✅ DPO model saved to ./outputs/dpo-checkpoint


train/epoch,▁▁▂▂▃▃▄▄▄▅▅▆▆▇▇██
train/global_step,▁▁▂▂▃▃▄▄▅▅▅▆▆▇▇██
train/grad_norm,█▁▃▁▁▁▁▁▁▁▁▁▁▁▁▁
train/learning_rate,▅███▇▇▆▅▅▄▃▃▂▂▁▁
train/logits/chosen,▆▃▇██▅▃▂▂▁▂▂▁▁▁▃
train/logits/rejected,▄▄▆██▄▂▂▂▁▁▁▁▁▁▂
train/logps/chosen,▇▇▇█▃▂▁▁▁▁▁▁▁▁▁▁
train/logps/rejected,█▇▇▇▃▁▁▁▁▁▁▁▁▁▁▁
train/loss,█▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
train/rewards/accuracies,▁███████████████
+8,...


In [9]:
# ── Evaluate DPO model ────────────────────────────────────────────────────────
from tqdm import tqdm
import torch, json

model.eval()
tokenizer.padding_side = "right"

SYSTEM_PROMPT = """You are a precise JSON extraction assistant. 
Given unstructured text, extract the requested fields and return ONLY valid JSON.
Do not include any explanation, markdown, or extra text. Output only the JSON object."""

def run_inference(instruction, input_text, model, tokenizer, max_new_tokens=256):
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": f"{instruction}\n\nText: {input_text}"}
    ]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(text, return_tensors="pt").to(model.device)
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=max_new_tokens,
                             temperature=0.1, do_sample=True,
                             pad_token_id=tokenizer.eos_token_id)
    return tokenizer.decode(out[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True).strip()

def is_valid_json(t):
    try: json.loads(t.strip()); return True
    except: return False

def exact_match(p, g):
    try: return json.loads(p.strip()) == json.loads(g.strip())
    except: return False

def field_coverage(p, g):
    try:
        po, go = json.loads(p.strip()), json.loads(g.strip())
        return sum(1 for k, v in go.items() if po.get(k) == v) / len(go)
    except: return 0.0

with open('./data/test_sft.json') as f:
    test_data = json.load(f)

results_dpo = []
for ex in tqdm(test_data[:100], desc="Evaluating DPO model"):
    pred = run_inference(ex['instruction'], ex['input_text'], model, tokenizer)
    results_dpo.append({
        'valid_json':     is_valid_json(pred),
        'exact_match':    exact_match(pred, ex['output']),
        'field_coverage': field_coverage(pred, ex['output']),
    })

n = len(results_dpo)
dpo_metrics = {
    'model': 'DPO Model (SFT + DPO)',
    'json_validity_rate':    sum(r['valid_json'] for r in results_dpo) / n,
    'exact_match_accuracy':  sum(r['exact_match'] for r in results_dpo) / n,
    'avg_field_coverage':    sum(r['field_coverage'] for r in results_dpo) / n,
    'n_samples': n
}

with open('./data/dpo_metrics.json', 'w') as f:
    json.dump(dpo_metrics, f, indent=2)

# Load all three for final comparison
with open('./data/baseline_metrics.json') as f:
    base = json.load(f)
with open('./data/sft_metrics.json') as f:
    sft = json.load(f)

print('\n' + '='*70)
print('📊 FINAL 3-WAY COMPARISON: Base → SFT → DPO')
print('='*70)
print(f'{"Metric":<35} {"Base":>10} {"SFT":>10} {"DPO":>10}')
print('-'*70)
for metric in ['json_validity_rate', 'exact_match_accuracy', 'avg_field_coverage']:
    print(f'{metric:<35} {base[metric]:>10.1%} {sft[metric]:>10.1%} {dpo_metrics[metric]:>10.1%}')
print('='*70)
print('✅ Saved dpo_metrics.json — run 05_final_comparison.ipynb for charts!')

Evaluating DPO model: 100%|██████████| 100/100 [04:16<00:00,  2.56s/it]


📊 FINAL 3-WAY COMPARISON: Base → SFT → DPO
Metric                                    Base        SFT        DPO
----------------------------------------------------------------------
json_validity_rate                      100.0%     100.0%     100.0%
exact_match_accuracy                     39.0%     100.0%     100.0%
avg_field_coverage                       80.2%     100.0%     100.0%
✅ Saved dpo_metrics.json — run 05_final_comparison.ipynb for charts!
